## 1. Raw Data
Loading the F1 Pit Strategy dataset.

In [ ]:
import pandas as pd
import numpy as np
import os

# Load data
data_path = 'playground-series-s6e5' if os.path.exists('playground-series-s6e5') else '.'
train = pd.read_csv(f'{data_path}/train.csv')
test = pd.read_csv(f'{data_path}/test.csv')
print(f'Train shape: {train.shape}')
print(f'Test shape: {test.shape}')

## 2. Data Validation
Checking for missing values and inspecting data types.

In [ ]:
print('Missing values in Train:')
print(train.isnull().sum())
print('\nData Types:')
print(train.dtypes)

## 3. Exploratory Data Analysis (EDA)
Visualizing the 2023 anomaly and driver pit stop frequencies to justify the new features.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Pit Rate by Year (Spotting the 2023 Anomaly)
year_stats = train.groupby('Year')['PitNextLap'].mean() * 100
plt.figure(figsize=(10, 5))
sns.barplot(x=year_stats.index, y=year_stats.values, palette='viridis')
plt.title('Pit Stop Rate (%) by Year - Spotting the 2023 Anomaly')
plt.ylabel('Pit Rate (%)')
plt.show()

# 2. Top 20 Drivers by Pit Frequency
driver_stats = train.groupby('Driver')['PitNextLap'].mean().sort_values(ascending=False).head(20)
plt.figure(figsize=(12, 6))
driver_stats.plot(kind='bar', color='salmon')
plt.title('Top 20 Drivers by Pit Frequency (Raw Signal)')
plt.ylabel('Pit Probablity')
plt.show()

## 4. Data Preprocessing & Feature Engineering
Implementing OOF Target Encoding for high-cardinality features and addressing the 2023 anomaly.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import StratifiedKFold

# Clean column names
train.columns = [c.replace(' ', '_').replace('(', '').replace(')', '').replace('[', '').replace(']', '') for c in train.columns]
test.columns = [c.replace(' ', '_').replace('(', '').replace(')', '').replace('[', '').replace(']', '') for c in test.columns]

# --- Experiment 2 Features (Progressive) ---
bins = [0, 3, 6, 10, 15, 20, 30, 40, 100]
train['TyreLifeBin'] = pd.cut(train['TyreLife'], bins=bins, labels=False)
test['TyreLifeBin'] = pd.cut(test['TyreLife'], bins=bins, labels=False)

train['PitWindowPressure'] = train['TyreLife'] * train['RaceProgress']
test['PitWindowPressure'] = test['TyreLife'] * test['RaceProgress']

train['Deg_diff'] = train.groupby(['Driver', 'Stint'])['Cumulative_Degradation'].diff().fillna(0)
test['Deg_diff'] = test.groupby(['Driver', 'Stint'])['Cumulative_Degradation'].diff().fillna(0)

# --- Experiment 3 Features ---

# 1. is_2023 Anomaly Flag
train['is_2023'] = (train['Year'] == 2023).astype(int)
test['is_2023'] = (test['Year'] == 2023).astype(int)

# 2. OOF Target Encoding for Driver
train['Driver_TE'] = 0
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for train_idx, val_idx in skf.split(train, train['PitNextLap']):
    X_t, X_v = train.iloc[train_idx], train.iloc[val_idx]
    means = X_t.groupby('Driver')['PitNextLap'].mean()
    train.loc[val_idx, 'Driver_TE'] = train.loc[val_idx, 'Driver'].map(means)

train['Driver_TE'] = train['Driver_TE'].fillna(train['PitNextLap'].mean())
test['Driver_TE'] = test['Driver'].map(train.groupby('Driver')['PitNextLap'].mean())
test['Driver_TE'] = test['Driver_TE'].fillna(train['PitNextLap'].mean())

# Ordinal Encode remaining categories
cat_cols = ['Driver', 'Compound', 'Race']
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
train[cat_cols] = encoder.fit_transform(train[cat_cols].astype(str))
test[cat_cols] = encoder.transform(test[cat_cols].astype(str))

print('Preprocessing and Feature Engineering complete.')

## 5. Model Training
Training the LightGBM baseline with the expanded feature set.

In [ ]:
import lightgbm as lgb

X = train.drop(['id', 'PitNextLap'], axis=1)
y = train['PitNextLap']

params = {
    'objective': 'binary',
    'metric': 'auc',
    'verbosity': -1,
    'device': 'gpu',
    'random_state': 42
}

try:
    # Test GPU
    temp_X = np.random.rand(10, X.shape[1])
    temp_y = np.random.randint(0, 2, 10)
    temp_data = lgb.Dataset(temp_X, label=temp_y)
    lgb.train(params, temp_data, num_boost_round=1)
    print('Using GPU for LightGBM')
except:
    params['device'] = 'cpu'
    print('Using CPU fallback')

## 6. Model Evaluation
Evaluating the model on a hold-out set.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

eval_model = lgb.LGBMClassifier(**params)
eval_model.fit(X_train, y_train)

val_preds = eval_model.predict_proba(X_val)[:, 1]
auc_score = roc_auc_score(y_val, val_preds)
print(f'ROC-AUC on hold-out set: {auc_score}')

## 7. Model Validation
Performing 5-Fold Stratified Cross-Validation.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
aucs = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_t, X_v = X.iloc[train_idx], X.iloc[val_idx]
    y_t, y_v = y.iloc[train_idx], y.iloc[val_idx]
    
    m = lgb.LGBMClassifier(**params)
    m.fit(X_t, y_t)
    p = m.predict_proba(X_v)[:, 1]
    auc = roc_auc_score(y_v, p)
    aucs.append(auc)
    print(f'Fold {fold+1} AUC: {auc}')

print(f'\nMean AUC: {np.mean(aucs):.5f} +/- {np.std(aucs):.5f}')

## 8. Model Deployment & Feedback
Saving final submission and model artifact.

In [ ]:
import pickle

final_model = lgb.LGBMClassifier(**params)
final_model.fit(X, y)

with open('lgbm_experiment3.pkl', 'wb') as f:
    pickle.dump(final_model, f)

test_X = test.drop(['id'], axis=1)
test_preds = final_model.predict_proba(test_X)[:, 1]
submission = pd.DataFrame({'id': test['id'], 'PitNextLap': test_preds})
submission.to_csv('submission_exp3.csv', index=False)
print('Saved artifacts for Experiment 3.')